# Train an IGNODE-compatible detector with RF-DETR (Small)

**Audience:** customers who want a transformer-based alternative to YOLOX. RF-DETR can outperform YOLO on cluttered scenes with many small overlapping objects, at the cost of slower training and inference.

**Output:** an ONNX model + sidecar JSON files ready for **Custom Model Upload** in your IGNODE workspace.

## When to choose RF-DETR over YOLOX
- ✅ Dense scenes (>20 objects per image)
- ✅ Small overlapping objects (cells under a microscope, tiny defects)
- ✅ You have ≥500 training images per class
- ❌ Limited training data (<100 images per class) → use YOLOX
- ❌ Real-time on edge devices → use YOLOX-nano or YOLOX-tiny

## Architecture decoder hint for IGNODE
RF-DETR exports use the `detr` decoder family in IGNODE UINF (`agent_runner/frameworks/detection_decoders/detr.py`). The sidecar declares `postprocess.family: 'detr'` — UINF parses logits + boxes from the DETR output shape and applies post-NMS filtering.

In [ ]:
# Step 0 — install RF-DETR + ONNX export deps. ~3 min on cold runtime.
# Need the [train,loggers] extras — `rfdetr` alone is inference-only;
# `model.train()` imports pytorch_lightning which ships in that extras
# group. Without it: `ImportError: RF-DETR training dependencies are
# missing`.
!pip install -q "rfdetr[train,loggers]" supervision==0.23.0 onnx==1.21.0 onnxruntime==1.23.2

In [ ]:
# IR-3.S.B — Settings: pick your dataset source + tune training knobs.
# Set ONE of DATASET_URL or DATASET_DIR. URL takes precedence.
# Leave both empty and the next cell will warn + stop.

# IR-3.S.F — Public download URL (.zip / .tar of the dataset, COCO layout).
#
# Quick test — copy this BCCD blood-cell-count sample (7.4 MB, 3 classes):
#   DATASET_URL = 'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/examples/datasets/bccd.coco.zip'
#
# Your own data: any public .zip / .tar download link works
# (Roboflow Universe 'Raw URL', GitHub Release asset, public S3 / Drive).
DATASET_URL = ''

# Google Drive folder path. Mount Drive separately if you want this.
# Example: '/content/drive/MyDrive/my-detection-dataset'
DATASET_DIR = ''

# IR-3.1.A.6 — Training knobs in the same Settings cell as DATASET_URL
# so the customer edits ONE cell to tune their run.
EPOCHS = 100         # RF-DETR converges faster than YOLOX; 100 is usually enough
BATCH_SIZE = 8       # RF-DETR is memory-hungry; lower than YOLOX's 16. Drop to 4 on smaller GPUs.

# IR-3.S.B.E — Early stopping knobs (moved out of the train() call so
# they sit next to EPOCHS/BATCH_SIZE in this Settings cell).
#   EARLY_STOPPING:           True to stop when val mAP plateaus,
#                             False to run for the full EPOCHS budget.
#   EARLY_STOPPING_PATIENCE:  number of val-eval ticks without
#                             improvement before training stops. 30
#                             (raised from 10) gives the transformer
#                             room to find a second wind on tougher
#                             datasets — RF-DETR's loss curve can flatten
#                             for a few epochs then drop sharply.
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 30

# Local scratch directory inside Colab — keep as-is.
WORKDIR = '/content/rfdetr-run'


In [ ]:
# IR-3.S.D — OPTIONAL Drive mount, gated so Run-all is safe.
# Only mounts Drive when the Settings cell points DATASET_DIR at a
# /content/drive path. URL customers + local-path customers skip
# this cell silently (no Drive auth popup, no Run-all stall).
import os

_needs_drive = (not DATASET_URL) and DATASET_DIR.startswith('/content/drive')
_already_mounted = os.path.ismount('/content/drive') or os.path.isdir('/content/drive/MyDrive')

if _needs_drive and not _already_mounted:
    from google.colab import drive
    drive.mount('/content/drive')
elif _needs_drive:
    print('Drive already mounted — skipping.')
else:
    print('Not using Drive (DATASET_URL set, or DATASET_DIR is a local path).')
    print('Skipping Drive mount.')


In [ ]:
# IR-3.S.B — Load dataset from whichever source the Settings cell set.
# Bails out loudly if BOTH DATASET_URL and DATASET_DIR are empty so the
# customer doesn't waste a 30-minute training run on an empty folder.
import os, pathlib, shutil, zipfile, tarfile, subprocess

if not DATASET_URL and not DATASET_DIR:
    raise SystemExit(
        '\n'
        '⚠️  Both DATASET_URL and DATASET_DIR are empty in the Settings cell.\n'
        '   Set ONE of them before running this cell:\n'
        '     • DATASET_URL — a public .zip / .tar download link\n'
        '     • DATASET_DIR — a path inside your mounted Google Drive\n'
        '   Then re-run this cell.'
    )

pathlib.Path(WORKDIR).mkdir(parents=True, exist_ok=True)

if DATASET_URL:
    print(f'Downloading from public URL: {DATASET_URL}')
    _archive = pathlib.Path('/content/_dataset_download')
    _archive.mkdir(parents=True, exist_ok=True)
    _dl_path = _archive / 'dataset.zip'
    subprocess.run(['curl', '-fsSL', '-o', str(_dl_path), DATASET_URL], check=True)
    print(f'Downloaded {_dl_path.stat().st_size:,} bytes — extracting…')
    _extracted = _archive / 'unpacked'
    if _extracted.exists():
        shutil.rmtree(_extracted)
    _extracted.mkdir()
    if zipfile.is_zipfile(_dl_path):
        with zipfile.ZipFile(_dl_path) as zf:
            zf.extractall(_extracted)
    elif tarfile.is_tarfile(_dl_path):
        with tarfile.open(_dl_path) as tf:
            tf.extractall(_extracted)
    else:
        raise SystemExit(f'Downloaded file is neither .zip nor .tar: {_dl_path}')
    DATASET_DIR = str(_extracted)
    print(f'Extracted to: {DATASET_DIR}')
else:
    if not pathlib.Path(DATASET_DIR).is_dir():
        raise SystemExit(
            f'❌ DATASET_DIR={DATASET_DIR!r} does not exist or is not a directory.\n'
            f'   If this is a Google Drive path, run the Drive mount cell below\n'
            f'   first — or set DATASET_URL instead.'
        )
    print(f'Using local/Drive folder: {DATASET_DIR}')

!ls -la "{DATASET_DIR}" | head -20


In [ ]:
# IR-3.S.G — Normalize ANY input format (VOC, YOLO, COCO) → COCO layout.
#
# RF-DETR's `model.train()` reads `DATASET_DIR/train/_annotations.coco.json`.
# Customers arrive with Pascal VOC (LabelImg / CVAT default) or YOLO
# (Ultralytics export) as often as COCO, so we transparently convert.
#
# This uses the SAME supervision library calls as IGNODE's platform-side
# ignode-format-converter sidecar — so a dataset that trains here will
# also train on-platform after upload to ML Factory. No format drift.
#
# Class-label order rule (IR-3.Y / IR-3.GG.H from the platform side):
# we take supervision's loader output `.classes` AS-IS and pass it
# straight into the COCO writer. NEVER sort, NEVER re-derive in parallel.
# That bug shipped a BCCD model where every prediction's label was
# rotated one slot — symptom: a banana labeled "orange".
#
# Image bytes are NEVER touched here. RGB/BGR channel order (IR-3.CC) is
# the sidecar's job — RF-DETR uses RGB (PyTorch ImageNet normalization),
# already declared in the sidecar cell below.

import json, pathlib, shutil
import supervision as sv

_root = pathlib.Path(DATASET_DIR)


def _detect_format(root: pathlib.Path) -> str:
    """Return 'coco' | 'yolo' | 'voc' | 'unknown' for the extracted root."""
    if list(root.rglob('_annotations.coco.json')) or list(root.rglob('instances_*.json')):
        return 'coco'
    if (root / 'data.yaml').is_file() or any(root.glob('*/data.yaml')):
        return 'yolo'
    if list(root.rglob('*.xml')):
        return 'voc'
    return 'unknown'


_fmt = _detect_format(_root)
print(f'Detected input format: {_fmt}')


if _fmt == 'coco':
    print('Already COCO — no conversion needed.')

elif _fmt == 'unknown':
    raise SystemExit(
        f'\nCould not detect dataset format in {DATASET_DIR}.\n'
        f'  Looking for ONE of:\n'
        f'    - COCO  (train/_annotations.coco.json + train/*.jpg)\n'
        f'    - VOC   (XML annotations paired with images per split)\n'
        f'    - YOLO  (data.yaml + images/<split>/ + labels/<split>/)\n'
        f'  Inspect the extracted folder and re-run.'
    )

else:
    # Convert to COCO. Write into a fresh sibling folder so we NEVER
    # mutate the customer's source data — re-running this cell stays
    # deterministic, and the original archive remains untouched.
    _converted = _root.parent / (_root.name + '_coco')
    if _converted.exists():
        shutil.rmtree(_converted)
    _converted.mkdir()
    print(f'Converting {_fmt} → COCO at: {_converted}')

    # Track classes per split to verify they match across splits. If they
    # differ, COCO's category_id namespace would conflict — bail early
    # rather than silently produce a corrupted dataset.
    _classes_seen = None

    if _fmt == 'voc':
        # Each split is a top-level folder containing both XMLs and JPGs.
        # Roboflow's VOC export uses this layout; LabelImg-only datasets
        # may need re-arranging by the customer before this cell.
        for _split_name in ['train', 'valid', 'test']:
            _split_in = _root / _split_name
            if not _split_in.is_dir():
                continue
            _out_split = _converted / _split_name
            _out_split.mkdir(parents=True, exist_ok=True)
            ds = sv.DetectionDataset.from_pascal_voc(
                images_directory_path=str(_split_in),
                annotations_directory_path=str(_split_in),
            )
            # IR-3.Y / IR-3.GG.H: take supervision's class list verbatim.
            # NEVER sort or re-derive.
            _classes = list(ds.classes)
            print(f'  {_split_name}: {len(ds)} images · classes = {_classes}')
            if _classes_seen is not None and _classes_seen != _classes:
                raise SystemExit(
                    f'Class mismatch between splits:\n'
                    f'  previous: {_classes_seen}\n'
                    f'  {_split_name}:   {_classes}\n'
                    f'Fix the source dataset so every split has the same classes in the same order.'
                )
            _classes_seen = _classes
            ds.as_coco(
                images_directory_path=str(_out_split),
                annotations_path=str(_out_split / '_annotations.coco.json'),
            )

    elif _fmt == 'yolo':
        # Find data.yaml (root or one level down — Roboflow's YOLOv8
        # export keeps it at root; some customer exports nest it).
        _yaml_path = _root / 'data.yaml'
        if not _yaml_path.is_file():
            _yaml_path = next(_root.glob('*/data.yaml'), None)
            if _yaml_path is None:
                raise SystemExit('YOLO format detected but no data.yaml found.')

        import yaml as _yaml_mod
        _spec = _yaml_mod.safe_load(_yaml_path.read_text())
        if not isinstance(_spec, dict):
            raise SystemExit(f'data.yaml is not a YAML mapping: {_yaml_path}')

        # YOLOv8 data.yaml typically:
        #   train: ../train/images   (or train/images)
        #   val:   ../valid/images
        #   names: ['ClassA', 'ClassB', ...]
        # Paths are relative to data.yaml's parent.
        _yaml_parent = _yaml_path.parent
        for _split_name in ['train', 'val', 'test']:
            _split_rel = _spec.get(_split_name)
            if not _split_rel:
                continue
            _img_dir = (_yaml_parent / _split_rel).resolve()
            if not _img_dir.is_dir():
                print(f'  {_split_name}: skipped (path does not exist: {_img_dir})')
                continue
            # Roboflow's YOLOv8 layout pairs images/<split>/ with labels/<split>/.
            _lbl_dir = _img_dir.parent.parent / 'labels' / _img_dir.name
            if not _lbl_dir.is_dir():
                print(f'  {_split_name}: skipped (labels folder missing: {_lbl_dir})')
                continue
            # RF-DETR convention: val split is named 'valid' on disk.
            _out_split = _converted / ('valid' if _split_name == 'val' else _split_name)
            _out_split.mkdir(parents=True, exist_ok=True)
            ds = sv.DetectionDataset.from_yolo(
                images_directory_path=str(_img_dir),
                annotations_directory_path=str(_lbl_dir),
                data_yaml_path=str(_yaml_path),
            )
            _classes = list(ds.classes)
            print(f'  {_split_name}: {len(ds)} images · classes = {_classes}')
            if _classes_seen is not None and _classes_seen != _classes:
                raise SystemExit(
                    f'Class mismatch between splits:\n'
                    f'  previous: {_classes_seen}\n'
                    f'  {_split_name}:    {_classes}\n'
                    f'Fix data.yaml so every split has identical class names in order.'
                )
            _classes_seen = _classes
            ds.as_coco(
                images_directory_path=str(_out_split),
                annotations_path=str(_out_split / '_annotations.coco.json'),
            )

    DATASET_DIR = str(_converted)
    print(f'\nConversion complete. DATASET_DIR is now: {DATASET_DIR}')
    print(f'Class order (will become CLASS_LABELS in the upload bundle):')
    for _i, _name in enumerate(_classes_seen or []):
        print(f'  [{_i}] {_name}')
    print()
    print('If this order surprises you, STOP and inspect your source dataset.')
    print('Wrong order here = every prediction mislabeled at deploy time.')
    !ls -la "{DATASET_DIR}"


In [ ]:
# Step 2 — train RF-DETR Small + scrape final mAP into results.json.
# Small balances accuracy + speed; RFDETRBase / RFDETRLarge swap in for
# better mAP at higher VRAM cost. All three auto-download COCO-pretrained
# weights on first instantiation — training is always fine-tuning.
#
# Parity with YOLOX notebooks: we tee training stdout to train.log,
# then parse RF-DETR's Rich-formatted Overall Metrics table for the
# final mAP@0.5:0.95 + mAP@0.5 and write results.json.
import sys, contextlib, re, json, pathlib
from rfdetr import RFDETRSmall

class _Tee:
    def __init__(self, *streams): self._s = streams
    def write(self, s):
        for st in self._s: st.write(s)
    def flush(self):
        for st in self._s: st.flush()

train_log = pathlib.Path(WORKDIR) / 'train.log'
pathlib.Path(WORKDIR).mkdir(parents=True, exist_ok=True)
with open(train_log, 'w') as _logf, contextlib.redirect_stdout(_Tee(sys.stdout, _logf)):
    model = RFDETRSmall()  # auto-downloads COCO-pretrained weights
    # IR-3.S.B.E — early_stopping / early_stopping_patience are sourced
    # from the Settings cell (EARLY_STOPPING / EARLY_STOPPING_PATIENCE).
    # Edit those ONCE at the top — no need to touch this cell.
    model.train(
        dataset_dir=DATASET_DIR,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        grad_accum_steps=4,          # effective batch = BATCH_SIZE * 4
        lr=1e-4,                     # transformer LR; lower than YOLO's 5e-3
        output_dir=WORKDIR,
        early_stopping=EARLY_STOPPING,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
    )

# Scrape final mAP from train.log.
# RF-DETR uses Rich tables; two parse paths combined for robustness:
#   1. "Best EMA mAP improved to <X>" plain-text lines  → AP@50:95
#   2. The "Val — Overall Metrics" table row (columns: 50:95 / 50 / 75
#      / @500 / F1 / Prec / Recall) — take the LAST occurrence after
#      stripping Rich's ANSI codes.
_log_text = train_log.read_text()
_ap_5095 = _ap_50 = None

for _m in re.finditer(r'Best EMA mAP improved to\s+([\d.]+)', _log_text):
    _ap_5095 = float(_m.group(1))

_ansi = re.compile(r'\x1b\[[0-9;]*m')
_clean = _ansi.sub('', _log_text)
for _line in _clean.splitlines():
    _row = re.match(
        r'^\s*[│|]\s*([\d.]+)\s*[│|]\s*([\d.]+)\s*[│|]\s*([\d.]+)\s*'
        r'[│|]\s*([\d.]+)\s*[│|]\s*([\d.]+)\s*[│|]\s*([\d.]+)\s*[│|]\s*([\d.]+)\s*[│|]\s*$',
        _line,
    )
    if _row:
        _ap_5095, _ap_50 = float(_row.group(1)), float(_row.group(2))

# Class list from COCO (same source as Cell 8).
_train_json = pathlib.Path(DATASET_DIR) / 'train' / '_annotations.coco.json'
_coco = json.loads(_train_json.read_text())
_classes = [str(c['name']) for c in sorted(_coco['categories'], key=lambda c: c.get('id', 0))]

_results = {
    'mAP_50':     _ap_50,
    'mAP_50_95':  _ap_5095,
    'epochs_run': EPOCHS,
    'batch_size': BATCH_SIZE,
    'classes':    _classes,
    'recipe':     {'optimizer': 'AdamW', 'lr_scheduler': 'cosine',
                   'lr': 1e-4, 'grad_accum_steps': 4,
                   'backbone': 'rfdetr_small', 'trainer': 'rfdetr'},
}
pathlib.Path(f'{WORKDIR}/results.json').write_text(json.dumps(_results, indent=2))
print()
print('Results summary:')
print(f'  mAP@0.5      = {_ap_50}')
print(f'  mAP@0.5:0.95 = {_ap_5095}')
print(f'  results.json written to {WORKDIR}/results.json')

In [ ]:
# Step 3 — export to ONNX.
#
# RFDETR.export() in 1.7+ writes into an `output_dir` (not a single
# file path) and figures out the filename itself. The torch-level
# args (`dynamic_axes`) are wrapped internally; surface what's there:
#   - output_dir       Directory to write the model.onnx into
#   - opset_version    ONNX opset (default 17; we want 18 for parity
#                      with IGNODE platform's pin)
#   - dynamic_batch    Bake a dynamic batch dimension into the graph
#   - format='onnx'    (also supports 'tflite' if you ever need it)
import pathlib
EXPORT_DIR = pathlib.Path(WORKDIR) / 'export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

model.export(
    output_dir=str(EXPORT_DIR),
    opset_version=18,
    dynamic_batch=True,
    format='onnx',
)

# Find whatever .onnx file rfdetr wrote inside EXPORT_DIR.
ONNX_OUT = next(EXPORT_DIR.rglob('*.onnx'), None)
assert ONNX_OUT is not None, f'No .onnx produced under {EXPORT_DIR}'
print('Exported:', ONNX_OUT, f'({ONNX_OUT.stat().st_size:,} bytes)')

In [ ]:
# IR-3.S.A — Step 4: write sidecars + auto-derive CLASS_LABELS.
# DO NOT hand-type class names. Sidecar values are derived from the
# trained model itself (input size from ONNX) and the renumbered COCO
# json (class names). The next cell smoke-tests the exported ONNX so
# you SEE the index→label map before uploading.
import json, shutil, pathlib

# 1. Class names — read from results.json (already phantom-filtered).
_results_path = pathlib.Path(WORKDIR) / 'results.json'
if _results_path.is_file():
    CLASS_LABELS = list(json.loads(_results_path.read_text()).get('classes') or [])
else:
    _train_json = pathlib.Path(DATASET_DIR) / 'train' / '_annotations.coco.json'
    _coco = json.loads(_train_json.read_text())
    # Filter Roboflow's phantom supercategory entry. Without this, a
    # phantom 'WBC' (id=0, supercategory='none') leaks into CLASS_LABELS,
    # rotating every prediction's label one slot at deploy time.
    _real = [c for c in _coco['categories'] if c.get('supercategory') != 'none']
    _real.sort(key=lambda c: c.get('id', 0))
    CLASS_LABELS = [str(c['name']) for c in _real]
print(f'Derived CLASS_LABELS:')
for i, name in enumerate(CLASS_LABELS):
    print(f'  [{i}] {name}')
print()
print('If this order surprises you, STOP and inspect the JSON before bundling.')

# 2. Input size — query the ONNX file we just exported. RF-DETR Small
# bakes 512x512; Base/Large differ. Always read from the model so the
# sidecar matches what UINF will feed at inference time.
import onnxruntime as _ort
_probe = _ort.InferenceSession(str(ONNX_OUT), providers=['CPUExecutionProvider'])
_inp_shape = _probe.get_inputs()[0].shape
_input_h = _inp_shape[2] if isinstance(_inp_shape[2], int) else 512
_input_w = _inp_shape[3] if isinstance(_inp_shape[3], int) else 512
print(f'ONNX input size: [{_input_h}, {_input_w}]')

preprocess_config = {
    'input_size':      [_input_h, _input_w],
    'mean':            [0.485, 0.456, 0.406],
    'std':             [0.229, 0.224, 0.225],
    'channel_order':   'RGB',
    'image_format':    'CHW',
    'rescale':         'imagenet',
    'resize_method':   'letterbox',
    'letterbox_color': [114, 114, 114],
    'postprocess': {
        'family':               'detr',
        'nms_required':         False,            # DETR is set-prediction; no NMS by design
        'confidence_threshold': 0.5,
        'num_queries':          300,
    },
    '_backbone': 'rfdetr_small',
    '_trainer':  'rfdetr',
}

out = pathlib.Path(WORKDIR) / 'upload-bundle'
out.mkdir(exist_ok=True)
shutil.copy(ONNX_OUT, out / 'model.onnx')
(out / 'preprocess_config.json').write_text(json.dumps(preprocess_config, indent=2))
(out / 'class_labels.json').write_text(json.dumps(CLASS_LABELS, indent=2))

print()
print('Upload bundle ready at:', out)
!ls -la {out}

In [ ]:
# IR-3.S.A — Step 4.5: ONNX smoke-test (DO NOT SKIP).
# Runs the freshly-exported ONNX on ONE validation image and prints
# the highest-confidence query's (class_index, label) pair. If the
# top prediction obviously contradicts the image's content, your
# class_labels.json is in the wrong order — fix BEFORE uploading.
import json, pathlib, numpy as np, onnxruntime as ort
from PIL import Image

_bundle = pathlib.Path(WORKDIR) / 'upload-bundle'
_labels = json.loads((_bundle / 'class_labels.json').read_text())
_sess   = ort.InferenceSession(str(_bundle / 'model.onnx'), providers=['CPUExecutionProvider'])
_inp    = _sess.get_inputs()[0].name
# Query the model's real input shape rather than hard-coding — Small
# bakes 512x512; Base/Large differ.
_inp_shape = _sess.get_inputs()[0].shape
_H = _inp_shape[2] if isinstance(_inp_shape[2], int) else 512
_W = _inp_shape[3] if isinstance(_inp_shape[3], int) else 512
print(f'Model expects [batch, 3, {_H}, {_W}] input.')

# Pick the first validation image. Change this path if your layout differs.
_val_dir   = pathlib.Path(DATASET_DIR) / 'valid'
_test_imgs = sorted([p for p in _val_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
assert _test_imgs, f'No images in {_val_dir} — adjust DATASET_DIR or the val split name.'
_img_path = _test_imgs[0]
print(f'Smoke-testing on: {_img_path.name}')

# Mirror Step 4 preprocess so the input matches what UINF will feed.
_pil = Image.open(_img_path).convert('RGB').resize((_W, _H), Image.BILINEAR)
_arr = (np.asarray(_pil, dtype=np.float32) / 255.0 - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
_arr = _arr.transpose(2, 0, 1)[None, ...]

_outs   = _sess.run(None, {_inp: _arr})
_logits = next((o for o in _outs if o.ndim == 3 and o.shape[-1] >= len(_labels)), None)
if _logits is None:
    print('WARN: could not find logits output — skipping smoke-test.')
else:
    _probs = 1.0 / (1.0 + np.exp(-_logits[0, :, :len(_labels)]))
    _best_query = int(_probs.max(axis=1).argmax())
    _best_class = int(_probs[_best_query].argmax())
    _best_score = float(_probs[_best_query, _best_class])
    print()
    print(f'Top-confidence query: idx={_best_class} → class_labels[{_best_class}] = {_labels[_best_class]!r}')
    print(f'Confidence: {_best_score:.3f}')
    print()
    print('Sanity check: open the image above. If the model thinks it is')
    print(f'a {_labels[_best_class]!r} but you can see it is something else, STOP and fix')
    print('class_labels.json before continuing.')

In [ ]:
# Bundle the upload files into ONE timestamped .zip, then ask before
# triggering the browser download. .zip lands at /content/<name>.zip
# (visible in Files panel parent of rfdetr-run/) — even if you skip
# the auto-download, you can still grab it from there.
import zipfile, pathlib
from datetime import datetime

STAMP       = datetime.now().strftime('%Y%m%d-%H%M')
BUNDLE_NAME = f'ignode-detector-rfdetr-{STAMP}'
ZIP_PATH    = pathlib.Path('/content') / f'{BUNDLE_NAME}.zip'
BUNDLE_SRC  = pathlib.Path(WORKDIR) / 'upload-bundle'

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in BUNDLE_SRC.iterdir():
        # Files go INSIDE a subfolder named after the zip — unzipping
        # creates ONE clearly-named folder, not 3 loose files spilled
        # into the customer's Downloads folder.
        zf.write(f, arcname=f'{BUNDLE_NAME}/{f.name}')

print('=' * 60)
print(f'  Bundle ready: {ZIP_PATH}')
print(f'  Contains: ' + ', '.join(p.name for p in BUNDLE_SRC.iterdir()))
print('=' * 60)

# Ask before downloading. Default is yes (just press Enter).
_choice = input('Download .zip now? (yes/no, default no): ').strip().lower()
if _choice in ('yes', 'y'):
    from google.colab import files
    print('Starting download...')
    files.download(str(ZIP_PATH))
else:
    print()
    print('Download skipped. To grab it later:')
    print(f'  open the Files panel (left sidebar) and right-click /content/{BUNDLE_NAME}.zip')


## Step 5 — upload to IGNODE

Same as YOLOX:
1. ML Factory → Models → **Upload custom model**
2. Drag the 3 files from `upload-bundle/` into the modal
3. Deploy to a UINF instance
4. Verify in Playground

## Troubleshooting
- **All predictions get filtered at threshold 0.5**: DETR's confidence is calibrated differently than YOLO. Try 0.3 or 0.2 in the Playground first; tighten once you see real predictions.
- **Wrong class names**: Check `class_labels.json` matches your dataset's COCO `categories[].name` order EXACTLY. UINF maps output index → list index.
- **Boxes outside image**: RF-DETR outputs normalized coords (0..1); UINF's `detr` decoder de-normalizes against `image_size`. If your input dims differ from 800×800, override `input_size` in `preprocess_config.json`.